# RetinaVessel AI — one-time official FIVES test v6

This notebook executes the frozen nine-checkpoint policy on the 200-image official FIVES test partition. It is commit-pinned, verifies every archive and checkpoint before opening test pixels, and cannot tune or select a model from test results.

**Protocol boundary:** running the evaluation cell changes the project record from `official_test_evaluated=false` to `true`. Attach the original FIVES dataset and a private Kaggle dataset containing the three unmodified run ZIPs, enable a GPU accelerator, and do not edit the plan or cells after results are visible.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "99920ac4cdddf039e0b70b3d7477f361084a45fb"
REPO = Path("/kaggle/working/retinavessel-ai-v6-code")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_commit == EXPECTED_COMMIT
os.chdir(REPO)
print("Pinned official-test code:", actual_commit)

In [ ]:
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest

assert torch.cuda.is_available(), "Enable a GPU accelerator before running v6"
DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
assert DATA_MOUNT.exists(), f"Missing attached FIVES dataset: {DATA_MOUNT}"
FIVES_ROOT = find_fives_root(DATA_MOUNT)
assert len(load_manifest(FIVES_ROOT, "train")) == 600
assert len(load_manifest(FIVES_ROOT, "test")) == 200  # paths and pairing only; pixels remain unopened
PLAN_PATH = REPO / "docs/fives_official_test_plan.json"
PLAN = json.loads(PLAN_PATH.read_text())
CHECKPOINT_ROOT = Path("/kaggle/working/frozen_checkpoints_v6")
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
archive_records = []
for archive_spec in PLAN["archives"]:
    matches = sorted(Path("/kaggle/input").rglob(archive_spec["filename"]))
    assert len(matches) == 1, f"Expected exactly one attached {archive_spec['filename']}, found {matches}"
    archive_path = matches[0]
    archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    assert archive_path.stat().st_size == archive_spec["bytes"]
    assert archive_hash == archive_spec["sha256"]
    with zipfile.ZipFile(archive_path) as archive:
        assert archive.testzip() is None
        prefix = f"seed{archive_spec['seed']}/"
        for checkpoint in [item for item in PLAN["checkpoints"] if item["seed"] == archive_spec["seed"]]:
            member = checkpoint["relative_path"].removeprefix(prefix)
            target = CHECKPOINT_ROOT / checkpoint["relative_path"]
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
            assert hashlib.sha256(target.read_bytes()).hexdigest() == checkpoint["sha256"]
    archive_records.append({**archive_spec, "attached_path": str(archive_path)})
print("CUDA device:", torch.cuda.get_device_name(0))
print("Verified archives:", [item["filename"] for item in archive_records])
print("Verified frozen checkpoints:", len(PLAN["checkpoints"]))
print("Official test pixels opened so far: 0")

In [ ]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), str(REPO / "backend")])
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, env=test_environment, check=True)

In [ ]:
RUN_ROOT = Path("/kaggle/working/retinavessel_v6_official_test")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ENVIRONMENT = RUN_ROOT / "environment"
ENVIRONMENT.mkdir(exist_ok=True)
pip_freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(ENVIRONMENT / "pip-freeze.txt").write_text(pip_freeze)
nvidia = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True, check=True)
(ENVIRONMENT / "nvidia-smi.csv").write_text(nvidia.stdout)
preflight = {
    "code_commit": actual_commit,
    "plan_sha256": hashlib.sha256(PLAN_PATH.read_bytes()).hexdigest(),
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
    "archives": archive_records,
    "official_test_evaluated": False,
}
(RUN_ROOT / "preflight.json").write_text(json.dumps(preflight, indent=2) + "\n")
assert not (RUN_ROOT / "official_test_report.json").exists()
print(json.dumps(preflight, indent=2))
print("NEXT CELL OPENS AND EVALUATES THE OFFICIAL TEST PIXELS.")

## Irreversible evaluation boundary

The next cell performs the single frozen evaluation. Resume is permitted only for checkpoint-complete partial output from this same plan; it does not change any model or setting.

In [ ]:
evaluation_command = [
    sys.executable, "-m", "retinavessel.official_test",
    "--data", str(DATA_MOUNT),
    "--plan", str(PLAN_PATH),
    "--checkpoint-root", str(CHECKPOINT_ROOT),
    "--out", str(RUN_ROOT),
    "--device", "cuda",
]
if (RUN_ROOT / "official_test_state.json").exists() and not (RUN_ROOT / "official_test_report.json").exists():
    evaluation_command.append("--resume")
evaluation_environment = os.environ.copy()
evaluation_environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run(evaluation_command, cwd=REPO, env=evaluation_environment, check=True)

In [ ]:
import pandas as pd

report = json.loads((RUN_ROOT / "official_test_report.json").read_text())
per_image = pd.read_csv(RUN_ROOT / "official_test_per_image.csv")
checkpoint_summary = pd.read_csv(RUN_ROOT / "official_test_checkpoint_summary.csv")
configuration_summary = pd.read_csv(RUN_ROOT / "official_test_configuration_summary.csv")
assert report["official_test_evaluated"] is True
assert report["official_test_image_count"] == 200
assert len(per_image) == 9 * 200
assert per_image.groupby("model_id")["image_id"].nunique().eq(200).all()
assert len(checkpoint_summary) == 9 and len(configuration_summary) == 3
assert report["selected_configuration_remains"] == "lite_robust"
assert report["individual_seed_selected"] is False
display(configuration_summary)
display(checkpoint_summary)

In [ ]:
result_files = sorted(path for path in RUN_ROOT.rglob("*") if path.is_file())
download_manifest = {
    "schema_version": 1,
    "official_test_evaluated": True,
    "code_commit": actual_commit,
    "plan_sha256": hashlib.sha256(PLAN_PATH.read_bytes()).hexdigest(),
    "files": [
        {"path": str(path.relative_to(RUN_ROOT)), "bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
        for path in result_files
    ],
}
(RUN_ROOT / "download_manifest.json").write_text(json.dumps(download_manifest, indent=2) + "\n")
ARCHIVE_BASE = Path("/kaggle/working/retinavessel_v6_official_test")
archive_path = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name))
print(f"Final official-test archive: {archive_path} ({archive_path.stat().st_size / 1024**2:.1f} MiB)")
print("Archive SHA-256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())
print("Save this executed notebook version and download the archive immediately.")

## Stop here

Do not change the frozen model decision or rerun with altered thresholds, post-processing, checkpoints, seeds or metrics. Download both the executed notebook and `retinavessel_v6_official_test.zip`; they are required to update the provenance ledger before external HRF and CHASE_DB1 validation.